<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/07_td_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R7 TD学習:Sarsa と Q学習(notebook)

本文:[`07-td-learning.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/07-td-learning.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

N_ROW, N_COL = 4, 12
S, A = N_ROW * N_COL, 4
START, GOAL = 3 * N_COL + 0, 3 * N_COL + 11
CLIFF = set(range(3 * N_COL + 1, 3 * N_COL + 11))
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # 0=上、1=右、2=下、3=左(R2 と同じ)
ARROWS = np.array(["↑", "→", "↓", "←"])

def next_state(s, a):
    r, c = divmod(s, N_COL)
    dr, dc = MOVES[a]
    r2, c2 = min(max(r + dr, 0), N_ROW - 1), min(max(c + dc, 0), N_COL - 1)
    return r2 * N_COL + c2

class CliffWalk:
    """Cliff Walking(R2 と同じ約束)。崖に入ると報酬 -100 でスタートに戻る(エピソードは続く)。"""
    def __init__(self, max_steps=None):
        self.max_steps = max_steps
    def reset(self, seed=None):
        self.s, self.t = START, 0
        return self.s, {}
    def step(self, a):
        s2 = next_state(self.s, a)
        reward = -1.0
        if s2 in CLIFF:
            reward, s2 = -100.0, START
        self.t += 1
        terminated = s2 == GOAL
        truncated = (not terminated) and self.max_steps is not None and self.t >= self.max_steps
        self.s = s2
        return s2, reward, terminated, truncated, {}

def epsilon_greedy_probs(Q_s, epsilon):            # R1 で作ったもの
    probs = np.full(len(Q_s), epsilon / len(Q_s))
    best = np.flatnonzero(Q_s == Q_s.max())
    probs[best] += (1 - epsilon) / len(best)
    return probs

def show_policy(Q, title):
    grid = ARROWS[Q.argmax(axis=1)].astype(object)
    for s in CLIFF: grid[s] = "C"
    grid[GOAL] = "G"
    print(title)
    print("\n".join(" ".join(row) for row in grid.reshape(N_ROW, N_COL)))

## 0. 環境:Cliff Walking

本文 §5 の 4×12 のグリッド。スタートはマス 36(左下)、ゴールはマス 47(右下)、崖はマス 37〜46。最短経路(崖のすぐ上)の収益は −13 である。

In [ ]:
env = CliffWalk()
s, _ = env.reset()
total = 0.0
for a in [0] + [1] * 11 + [2]:                     # 上 → 右11回 → 下
    s, r, terminated, truncated, _ = env.step(a)
    total += r
check("崖のすぐ上を通ると、13ステップ・収益 -13 でゴール", [total, terminated], [-13.0, True])
env.reset(); s, r, *_ = env.step(1)
check("スタートから右へ進むと崖:報酬 -100 でスタートに戻る", [s, r], [START, -100.0])

## 1. TD目標

本文 §2 の表のとおり、3つの手法の TD目標を計算する関数を書く。`terminated` なら、TD目標は `r` だけである。

**手計算**:本文の確認課題1・2( $Q(s, a) = -4$ 、 $r = -1$ 、 $Q(s', \cdot) = (-3, -6, -2, -5)$ 、 $a' = 1$ 、 $\alpha = 0.5$ 、 $\gamma = 1$ 、 $\varepsilon = 0.1$)を解く。

- Sarsa:(ここに書く)
- Q学習:(ここに書く)
- Expected Sarsa の TD目標:(ここに書く)

In [ ]:
def td_target(method, Q, r, s2, a2, terminated, gamma=1.0, epsilon=0.1):
    if terminated:
        return r
    if method == "sarsa":
        raise NotImplementedError("TODO: r + gamma * Q[s2, a2]")
    if method == "qlearning":
        raise NotImplementedError("TODO: r + gamma * Q[s2] の最大値")
    if method == "expected_sarsa":
        raise NotImplementedError("TODO: r + gamma * (epsilon_greedy_probs(Q[s2], epsilon) と Q[s2] の内積)")
    raise ValueError(method)

In [ ]:
Qt = np.zeros((2, 4)); Qt[0, 0] = -4.0; Qt[1] = [-3, -6, -2, -5]
alpha = 0.5
for method, expected in [("sarsa", -5.5), ("qlearning", -3.5)]:
    target = td_target(method, Qt, -1.0, 1, 1, False)
    check(f"{method}:更新後の Q(s,a)", Qt[0, 0] + alpha * (target - Qt[0, 0]), expected)
check("expected_sarsa の TD目標", td_target("expected_sarsa", Qt, -1.0, 1, 1, False), -1 + (0.025 * (-3 - 6 - 5) + 0.925 * -2))
check("terminated なら r だけ", td_target("qlearning", Qt, -1.0, 1, 1, True), -1.0)

## 2. 学習のループ

1エピソードの中で、次をくり返す(Sarsa は次の行動 $a'$ を先に選んでおく必要があるので、3つの手法で同じ書き方にしている)。

1. 行動 `a` で1ステップ進める → `s2, r, terminated, truncated`
2. `s2` での次の行動 `a2` を ε-greedy で選ぶ
3. TD目標を計算し、`Q[s, a]` を本文の式 (1) で更新する
4. `s, a = s2, a2`

In [ ]:
def run(method, alpha=0.5, epsilon=0.1, gamma=1.0, n_episodes=500, seed=0, env=None):
    env = env or CliffWalk()
    rng = np.random.default_rng(seed)
    Q = np.zeros((S, A))
    returns = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s, _ = env.reset()
        a = int(rng.choice(A, p=epsilon_greedy_probs(Q[s], epsilon)))
        done = False
        while not done:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = int(rng.choice(A, p=epsilon_greedy_probs(Q[s2], epsilon)))
            raise NotImplementedError("TODO: td_target で TD目標を計算し、Q[s, a] を alpha だけ動かす")
            s, a = s2, a2
            returns[ep] += r
            done = terminated or truncated
    return Q, returns

In [ ]:
def greedy_return(Q, max_steps=100):
    # 学習後の評価:探索をやめて(greedy で)動かしたときの収益
    env = CliffWalk(max_steps=max_steps)
    s, _ = env.reset(); total, done = 0.0, False
    while not done:
        s, r, terminated, truncated, _ = env.step(int(Q[s].argmax()))
        total += r; done = terminated or truncated
    return total

Q_q, ret_q = run("qlearning", seed=0)
show_policy(Q_q, "Q学習の greedy な方策")
check("Q学習:学習後に greedy で動かすと、最短経路(収益 -13)", greedy_return(Q_q), -13.0)

## 3. 3つの手法を比べる

**予想**:学習中(ε-greedy で行動中)の1エピソードあたりの収益は、Sarsa と Q学習のどちらが高いか。学習後に greedy で動かしたときはどうか。

- 予想:(ここに書く)

seed を変えて 20 回ずつ学習し、学習曲線(学習中の収益の移動平均)を、seed で平均して比べる。少し時間がかかる(数十秒)。

In [ ]:
METHODS = {"Sarsa": "sarsa", "Q-learning": "qlearning", "Expected Sarsa": "expected_sarsa"}
N_RUNS = 20
curves, final_Q = {}, {}
for name, m in METHODS.items():
    runs = [run(m, seed=k) for k in range(N_RUNS)]
    curves[name] = np.array([ret for _, ret in runs])
    final_Q[name] = [Q for Q, _ in runs]

def moving_average(x, w=10):
    return np.convolve(x, np.ones(w) / w, mode="valid")

for name, c in curves.items():
    plt.plot(moving_average(c.mean(axis=0)), label=name)
plt.ylim(-120, 0); plt.xlabel("episode"); plt.ylabel(f"return during learning ({N_RUNS}-run mean)")
plt.legend(); plt.grid(); plt.show()
for name in METHODS:
    online = curves[name][:, -100:].mean()
    evals = np.array([greedy_return(Q) for Q in final_Q[name]])
    reached = evals > -100                          # 100ステップで着かない = 方策が同じマスを回っている
    print(f"{name:15s}: 学習中の収益(最後の100エピソード){online:7.1f}   学習後の greedy:ゴールに着いた {reached.sum()}/{N_RUNS} 回、着いたときの収益の中央値 {np.median(evals[reached]):.0f}")

In [ ]:
show_policy(final_Q["Sarsa"][0], "Sarsa の greedy な方策")
print()
show_policy(final_Q["Q-learning"][0], "Q学習の greedy な方策")

Sarsa は崖から離れた経路、Q学習は崖のすぐ上の経路を学ぶ(本文 §4〜5)。

Sarsa の greedy な方策は、seed によっては、ゴールに着かずに同じマスを回ることがある。Sarsa は ε-greedy で行動したときの価値を学ぶので、あまり訪れないマスの価値が十分に学ばれず、そこで greedy に選ぶと行き止まりになることがあるためである。学習中の収益と、学習後の評価の収益が違う量であることが分かる(本文 §6)。

## 4. 「正解」と比べる:価値反復の $Q^\ast$

この環境は決定的で、モデルを作れる。R4 の価値反復で、最適な行動の価値 $Q^\ast$ を計算し、Q学習の結果と比べる(本文 §6)。

**実装**:Cliff Walking の `P`(形 `(S, A, S)`)と `R`(形 `(S, A)`)を作る。崖に入る行動は「報酬 −100 でスタートへ」、ゴールでは「その場にとどまり報酬 0」とする。

In [ ]:
def build_model():
    P, R = np.zeros((S, A, S)), np.zeros((S, A))
    for s in range(S):
        for a in range(A):
            if s == GOAL:
                P[s, a, s] = 1.0
                continue
            raise NotImplementedError("TODO: s2 = next_state(s, a)。崖なら P[s,a,START]=1, R=-100。そうでなければ P[s,a,s2]=1, R=-1")
    return P, R

In [ ]:
P, R = build_model()
check("遷移確率の和は 1", P.sum(axis=2), np.ones((S, A)))

def value_iteration_Q(P, R, gamma=1.0, tol=1e-10):
    Q = np.zeros(R.shape)
    while True:
        Q_new = R + gamma * P @ Q.max(axis=1)        # ベルマン最適方程式(本文の式 (3))を代入する
        if np.abs(Q_new - Q).max() < tol:
            return Q_new
        Q = Q_new

Q_star = value_iteration_Q(P, R)
check("スタートの最適な価値は -13", Q_star[START].max(), -13.0)
path, s = [], START
while s != GOAL:
    path.append(s); s = next_state(s, int(Q_star[s].argmax()))
opt_actions = [int(Q_star[s].argmax()) for s in path]
check("最短経路の上で、Q学習の greedy な行動 = 最適な行動", [int(Q_q[s].argmax()) for s in path], opt_actions)
print("最短経路の上で、|Q学習の Q - Q*| の最大(選んだ行動):", np.abs([Q_q[s, a] - Q_star[s, a] for s, a in zip(path, opt_actions)]).max().round(4))
print("最短経路の上で、Sarsa の Q(選んだ行動)- Q*:", np.round([final_Q['Sarsa'][0][s, a] - Q_star[s, a] for s, a in zip(path, opt_actions)], 1))

Q学習の $Q$ は、最短経路の上で $Q^\ast$ とほぼ一致する。この環境は決定的なので、TD目標のばらつきが小さく、Q学習はベルマン最適方程式の解にそのまま近づく(本文 §3)。Sarsa の $Q$ は、崖のそばで $Q^\ast$ より小さい。探索で崖に落ちる分を価値に入れているからである。

## 5. 条件を変える

1. $\varepsilon$ を 0.01、0.1、0.3 に変えると、Sarsa の学ぶ経路と、学習中の収益はどう変わるか
2. ステップサイズ $\alpha$ を 0.1、0.5、0.9 に変えると、Q学習の学習曲線はどう変わるか
3. 時間切れを入れ(`CliffWalk(max_steps=20)`)、TD目標で `truncated` を `terminated` と同じに扱うと、何が起きるか(本文 §2 の WARNING)

In [ ]:
# 1. epsilon と Sarsa の経路(greedy で動かしたときのステップ数で見る)
for eps in [0.01, 0.1, 0.3]:
    res = [run("sarsa", epsilon=eps, seed=k) for k in range(5)]
    ev = np.array([greedy_return(Q) for Q, _ in res])
    print(f"1. eps = {eps:4}: Sarsa の学習後の greedy の収益(着いたもの){ev[ev > -100].tolist()}   学習中の収益(最後の100) {np.mean([r[-100:].mean() for _, r in res]):7.1f}")
# 2. alpha と学習の速さ
for alpha in [0.1, 0.5, 0.9]:
    rets = np.array([run("qlearning", alpha=alpha, seed=k)[1] for k in range(5)])
    print(f"2. alpha = {alpha}: Q学習の学習中の収益 最初の50 {rets[:, :50].mean():7.1f}   最後の100 {rets[:, -100:].mean():7.1f}")

# 3. truncated を terminated と同じに扱った場合
def run_wrong(alpha=0.5, epsilon=0.1, n_episodes=500, seed=0):
    env = CliffWalk(max_steps=20)
    rng = np.random.default_rng(seed); Q = np.zeros((S, A))
    for ep in range(n_episodes):
        s, _ = env.reset(); a = int(rng.choice(A, p=epsilon_greedy_probs(Q[s], epsilon))); done = False
        while not done:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = int(rng.choice(A, p=epsilon_greedy_probs(Q[s2], epsilon)))
            target = td_target("qlearning", Q, r, s2, a2, terminated or truncated)   # 間違い:打ち切りも「終わり」にしている
            Q[s, a] += alpha * (target - Q[s, a]); s, a = s2, a2; done = terminated or truncated
    return Q
Q_ok, _ = run("qlearning", env=CliffWalk(max_steps=20), seed=0)
Q_ng = run_wrong(seed=0)
print(f"3. スタートの価値(正解 -13):正しい扱い {Q_ok[START].max():.2f}   打ち切りを終わりと扱う {Q_ng[START].max():.2f}")

## 6. 研究室のライブラリ `cpai_lab` との関係

`cpai_lab` の `compute_td_error` は、この単元の TD誤差(TD目標 − `Q[s, a]`)を計算する。`train_td` は、Gymnasium の約束で作った環境(`observation_space.n` と `action_space.n` を持つもの)で、この単元と同じ学習のループを実行する。`GridMazeEnv` の落とし穴(`T`)で崖のある迷路を作り、Sarsa と Q学習の違いを確かめる(落とし穴に入るとエピソードが終わる)。

In [ ]:
# 研究室のライブラリ cpai_lab を入れる(Colab では、ランタイムをつなぐたびに必要)
%pip install -q git+https://github.com/oit-cpai/lab-core.git

In [ ]:
from cpai_lab import compute_td_error, train_td, GridMazeEnv
from cpai_lab.envs.grid_maze import grid_from_text

rng = np.random.default_rng(1)
Qr = rng.normal(size=(5, 4))
for m in ["sarsa", "qlearning", "expected_sarsa"]:
    got = compute_td_error(m, Qr, 0, 2, -1.0, 3, 1, False, epsilon=0.1, gamma=0.9)
    check(f"compute_td_error({m}) = 自分の TD目標 - Q[s,a]", got, td_target(m, Qr, -1.0, 3, 1, False, 0.9, 0.1) - Qr[0, 2])

grid, start, goal, traps = grid_from_text("""
##############
#............#
#............#
#............#
#STTTTTTTTTTG#
##############
""")
lab_env = GridMazeEnv(grid=grid, start=start, goal=goal, traps=traps, step_reward=-1.0, goal_reward=-1.0, trap_reward=-100.0)
for m in ["sarsa", "qlearning"]:
    Q_lab, _ = train_td(lab_env, m, alpha=0.5, epsilon=0.1, gamma=1.0, num_episodes=500, seed=0)
    s, _ = lab_env.reset(); path = [lab_env.obs_to_pos(s)]
    for _ in range(50):
        s, r, terminated, truncated, _ = lab_env.step(int(Q_lab[s].argmax())); path.append(lab_env.obs_to_pos(s))
        if terminated: break
    print(f"{m:10s}: greedy で {len(path) - 1} ステップ。通った行の番号 {sorted(set(p[0] for p in path))}(4 はスタート・崖・ゴールの行、3 は崖のすぐ上の行)")

## 7. 説明(自分の言葉で書く)

1. Q学習の更新式は、ベルマン最適方程式とどう関係しているか。「期待値」「標本」「平均」という言葉を使って説明する
   - (ここに書く)
2. Cliff Walking で、Sarsa と Q学習が違う経路を学ぶのはなぜか
   - (ここに書く)
3. 学習中の収益と、学習後に greedy で動かした収益は、何が違うか。研究で結果を書くときに、どちらを使ったかを書く必要があるのはなぜか
   - (ここに書く)